# P04 e I01. Rendimiento y traza finita

23 y 24 de octubre de 2026, sesiones 7 y 8

Preparado por el PhD Esteban Hernández, CyberColombia.

Estos ejercicios usan tres muestras reales de 1.000 filas. Las primeras filas sirven para probar código, sin representatividad estadística. Los archivos originales permanecen intactos. No se requiere GPU ni una cuenta externa. Instalar requirements.txt y requirements_pqrs.txt y seleccionar el kernel del mismo entorno.


**Entorno:** Ubuntu-26.04 sobre WSL. Clonar siempre `main` en `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. Activar `.venv` desde la raíz y abrir Jupyter desde `kit/`. Seguir [la guía WSL](../../docs/instalacion-wsl.md).

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "kit", Path.cwd() / "big-data-postgraduate" / "kit"]
KIT = next((p.resolve() for p in candidates if (p / "pqrs_talleres.py").exists()), None)
assert KIT is not None, "Abrir desde bigdata/, la raíz del repositorio, kit/ o kit/Notebooks/"
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=KIT / "salidas" / "notebooks", repeticiones=3)
a.out.mkdir(parents=True, exist_ok=True)


In [2]:
lab.formatos(a)
mediciones = lab.benchmark(a)
assert len(mediciones) == 15
assert all(x["filas_entrada"] == 3000 and x["equivalente"] for x in mediciones)
import pandas as pd
pd.read_csv(a.out / "benchmark_pqrs_resumen.csv")

[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]
{'filas': 3000, 'columnas_producto': 16, 'columnas_original': 38, 'bytes_parquet': 65473, 'bytes_csv_proyeccion': 257499, 'particiones_archivos': 8, 'firma_contenido': '27186351908317914318628', 'bloque': 500, 'alcance': 'muestra', 'particion': ['anio', 'mes_num'], 'originales_modificados': False}
              motor  mediana_segundos  rss_max_muestreado_bytes
             duckdb          3.164907                 146935808
         duckdb_csv          3.169838                 151769088
duckdb_particionado          3.412005                 148774912
             pandas          4.519039                 145952768
             polars          4.376425                 173195264


,motor,mediana_segundos,rss_max_muestreado_bytes
0,duckdb,3.164907,146935808
1,duckdb_csv,3.169838,151769088
2,duckdb_particionado,3.412005,148774912
3,pandas,4.519039,145952768
4,polars,4.376425,173195264


## Extensión con el volumen completo, dentro del bloque guiado
Los tres CSV completos suman 1,816 GB. El docente usa la copia validada o descarga un corte por vez. Reservar 8 GB para archivos y salidas, más el entorno instalado. Con 8 GB de RAM mantener pandas en muestras. La consulta completa fue comprobada con 2.444.766 filas.

`python pqrs_talleres.py formatos --completo --datos data/pqrs_completos`

`python pqrs_talleres.py benchmark --completo --repeticiones 3`

Registrar máquina, versiones y alcance del tiempo. Cada repetición arranca un proceso y exporta un pequeño resultado. La RSS se sondea cada 10 ms. La caché del sistema puede persistir.

## I01. Reintentos y atraso, sesión 8
NASA aporta observaciones diarias reales. La reproducción altera el orden de entrega y repite una observación. La regla Python de tres días permite razonar sobre la traza; no representa toda la semántica de Spark.

In [3]:
r = lab.eventos(a)
assert (r["entregas"], r["registros_reales_distintos"], r["aceptados"], r["duplicados"], r["tardios"]) == (13,12,11,1,1)
assert abs(r["suma_aceptada_mm"] - 41.44) < 1e-8
pd.read_csv(a.out / "traza_eventos_nasa.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw/nasa.json'

## Entrega
Informe experimental con equivalencia y límites, más traza explicada. Proponer cómo corregir la llegada tardía y contrastar después con E11 en Spark. No sumar las series mensuales de PQRS con lluvia ni fabricar tiempos de evento para reportes.